# Data Collection and Processing

------------------------------------------------------------

## 1. Scraping Replays from Pokémon Showdown

<span style="color:white;background:darkgreen;padding:0px 3px;border-radius:2px">Note:</span> For simplicity, here we assume that `pwd` is the repo base directory "`/`".

To scrape `gen9-randombattle` replay JSONs into a directory, run
```zsh
python tools/scraper.py [/your/dir]
```

Other options and usage details can be found by using
```zsh
python tools/scraper.py -h
```

## 2. Parsing battles into `pandas.DataFrame` and removing custom-rule battles

Naturally, the following can be modified and run in different directories.

<u>(3a) Compiling battle data</u>: 

In [ ]:
from tools.battle import *
from tools.bat_to_list import battle_to_list

# NOTE: Unzip the folder(s) in /data/replays to run this, or change to your desired directory
replay_dir = Path("../data/replays/test_data_replays/") 

# ===========================
DATA = []
customs = []
errs = []

for replay in replay_dir.glob("*.json") : 
    try : 
        with replay.open() as file :
            replay_json = json.load(file)
        bat = Battle(replay_json, parse=True)
        
        if not bat.custom_ruleQ : 
            DATA.append(battle_to_list(bat))
        else : 
            customs.append(replay.name)
    except : 
        print(f"error with {replay.name}")
        errs.append(replay.name)
        continue

print(customs)
print(errs)

[]
[]


<u>(3b) Delete any replays having custom rules</u>: 

In [ ]:
import os
for replay in customs : 
    os.remove(replay_dir / replay.name)

<u>(3c) Make `DATA` into `pandas.DataFrame` and save</u>: 

In [11]:
import pandas as pd 

with open('../data/data_col_names.txt', 'r') as file:
    col_names = eval(file.read())

df = pd.DataFrame(DATA, columns=col_names)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4391 entries, 0 to 4390
Columns: 299 entries, format to M26_off
dtypes: bool(13), float64(1), int64(126), str(159)
memory usage: 14.9 MB


In [ ]:
with open('../data/test_data_cleaned.csv', 'w') as file:
    file.write(df.to_csv(index=False))

<u>(3d) Testing read-in</u>:

In [13]:
df = pd.read_csv("../data/test_data_cleaned.csv")
df.info() # testing

<class 'pandas.DataFrame'>
RangeIndex: 4391 entries, 0 to 4390
Columns: 299 entries, format to M26_off
dtypes: bool(13), float64(1), int64(126), str(159)
memory usage: 14.9 MB
